# Load genomic data (relevant columns only), do one hot encoding, save for training/validation

In [1]:
import pandas as pd
import polars as pl
import numpy as np
import random

import json
import yaml
import shutil

import random
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

In [2]:
!pwd

/project/ich248_uksr/DMYTRO/ZCOR/RESEARCHINGS/COLORADO_BIOBANK/ILD/COMBINE_ZEBRA_AND_GENDRIVERS


In [3]:
GENDIR = "/project/ich248_uksr/IXC/LSM_genome"

In [4]:
with open(f"{GENDIR}/genomicdata.csv", 'r') as f:
    RAW_COLS = f.readline().replace('\n','').split(',')

In [5]:
DRIVERS = pd.read_csv(f"{GENDIR}/biological_drivers.csv")
DRIVERS_2 = pd.read_csv(f"MORE_LOCI.csv")
print(DRIVERS_2.shape)
# Both direct SNP names and chr:basepair formats to look for column_names
SNPS = set(DRIVERS_2.SNP) | set([f"{r.CHR}:{r.BP}" for _, r in DRIVERS_2.iterrows()])
NEW_DRIVER_COLUMNS = [i for i in RAW_COLS if i[:-2] in SNPS] # All columns end with _{somenucleotide}
print(len(NEW_DRIVER_COLUMNS))

(3113, 7)
148


In [6]:
DRIVERS_2.sample(4)

,source_sheet,SNP,CHR,BP,P,Base,Gene
2154,"gwas p<7E-4, keep reseq SNPs",rs146797502,19,18301023,0.000191,1,MPV17L2
1467,gwas_wgs_prs_7E-4_SNPs,rs10502319,18,4019209,0.000647,1,DLGAP1
2540,"gwas p<7E-4, keep reseq SNPs",rs11781619,8,96017323,0.000393,1,NDUFAF6
1743,"gwas p<7E-4, keep reseq SNPs",rs17418062,4,87336776,0.000018,1,MAPK10


In [7]:
"""
    Find FID in patient_ids
"""
ZEBRA_PREDS = pd.read_parquet(f"PREDICTIONS_104W_PRED_WINDOW.parquet")

In [8]:
ZEBRA_PREDS.head(2)

,patient_id,predicted_risk,error_code
0,6.682883e+09,0.235006,
1,6.824091e+09,0.402156,


### Only load the columns you need - would be much faster with .parquet file

#### Loading loci from both csvs I received

#### this cell takes some time and ~65GB of RAM

In [9]:
try:
    GEN_DATA = pl.read_csv(
        f"{GENDIR}/genomicdata.csv", columns = ["FID"] + list(set(NEW_DRIVER_COLUMNS + list(DRIVERS['column_name'])))
    )
except ColumnNotFoundError:
    print("ColumnNotFoundError")

In [10]:
GD = GEN_DATA.to_pandas().rename(columns = {'FID': 'patient_id'})

In [11]:
GD.shape

(19651, 172)

In [12]:
GD[list(GD.columns)[9]].value_counts()

rs699240_T
1.0    9493
2.0    6229
0.0    3826
Name: count, dtype: int64

In [13]:
GD.iloc[:4,:4]

,patient_id,rs3131520_C,rs55993474_A,rs2375892_T
0,1230142395,1.0,2.0,1.0
1,6674359887,2.0,2.0,1.0
2,6802160313,2.0,2.0,2.0
3,6489473597,2.0,2.0,0.0


In [14]:
# One-hot encode all genotype columns, forcing categories 0, 1, 2
ID = GD[["patient_id"]]

X = GD.drop(columns="patient_id").apply(
    lambda x: pd.Categorical(x, categories=[0, 1, 2])
)

X = pd.get_dummies(X, dtype=int)

GD_OHE = pd.concat([ID, X], axis=1)

In [15]:
GD_OHE[GD_OHE.patient_id.isin(set(ZEBRA_PREDS.patient_id))].iloc[:7,:7]

,patient_id,rs3131520_C_0,rs3131520_C_1,rs3131520_C_2,rs55993474_A_0,rs55993474_A_1,rs55993474_A_2
0,1230142395,0,1,0,0,0,1
2,6802160313,0,0,1,0,0,1
3,6489473597,0,0,1,0,0,1
4,5491627298,0,1,0,0,0,1
6,6810276030,0,0,1,0,0,1
8,6427269461,0,1,0,0,0,1
11,5997834115,0,1,0,0,0,1


In [16]:
print(GD_OHE.shape)
display(GD_OHE.head(3))
GD_OHE.to_csv("ILD_TOP_DRIVERS_DATA.csv", index = False)

(19651, 514)


,patient_id,rs3131520_C_0,rs3131520_C_1,rs3131520_C_2,rs55993474_A_0,rs55993474_A_1,rs55993474_A_2,rs2375892_T_0,rs2375892_T_1,rs2375892_T_2,...,rs4811028_G_2,rs12626788_C_0,rs12626788_C_1,rs12626788_C_2,rs75764565_C_0,rs75764565_C_1,rs75764565_C_2,rs45619034_G_0,rs45619034_G_1,rs45619034_G_2
0,1230142395,0,1,0,0,0,1,0,1,0,...,1,0,0,1,0,0,1,0,0,1
1,6674359887,0,0,1,0,0,1,0,1,0,...,1,1,0,0,0,0,1,0,1,0
2,6802160313,0,0,1,0,0,1,0,0,1,...,1,0,1,0,0,0,1,0,0,1
